# Training — All 7 Ablation Variants — PyTorch
### MedAI Diagnose | CNN + NLP + PEPA | RTX 5050


In [16]:
# Cell 1 — Imports and GPU Setup
import os, time, pickle, shutil, warnings, gc, random
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import torchvision.models as models
import torchvision.transforms as transforms

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {device}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Deterministic setup for fair A6 vs A7 comparisons
DETERMINISTIC = True
if DETERMINISTIC:
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    print('Deterministic mode: ON')
else:
    torch.backends.cudnn.benchmark = True
    print('Deterministic mode: OFF (faster)')

PyTorch  : 2.10.0+cu128
Device   : cuda
GPU      : NVIDIA GeForce RTX 5050 Laptop GPU
VRAM     : 8.5 GB
Deterministic mode: ON


In [17]:
# Cell 2 — Configuration
BASE      = r'C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose'
TEXT_CSV  = os.path.join(BASE, 'dataset', 'cleaned', 'text', 'master_text_dataset.csv')
IMG_DIR   = os.path.join(BASE, 'dataset', 'cleaned', 'images')
ARTIFACTS = os.path.join(BASE, 'artifacts')
RESULTS   = os.path.join(BASE, 'results')
FIG_DIR   = os.path.join(BASE, 'results', 'figures')

for d in [ARTIFACTS, RESULTS, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

TFIDF_DIM  = 500
EMBED_DIM  = 128
IMG_SIZE   = (224, 224)
BATCH_SIZE = 64
EPOCHS     = 40
LR         = 1e-3
PATIENCE   = 7
SEED       = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

print(f'TEXT_CSV exists : {os.path.exists(TEXT_CSV)}')
print(f'IMG_DIR  exists : {os.path.isdir(IMG_DIR)}')
print(f'EPOCHS          : {EPOCHS}')

TEXT_CSV exists : True
IMG_DIR  exists : True
EPOCHS          : 40


In [18]:
# Cell 3 — Load Dataset + Image Path Matching
#
# COMPREHENSIVE MAPPING: all IMG_DIR folder names are mapped.
# Goal: maximize real-image rows and reduce zero tensors.

print('Loading dataset...')
df = pd.read_csv(TEXT_CSV)
print(f'  Shape    : {df.shape}')
print(f'  Diseases : {df["disease"].nunique()}')

encoder = LabelEncoder()
y_all   = encoder.fit_transform(df['disease'])
NUM_CLS = len(encoder.classes_)
print(f'  Classes  : {NUM_CLS}')

print('Fitting TF-IDF...')
vectorizer = TfidfVectorizer(max_features=TFIDF_DIM, ngram_range=(1,2), sublinear_tf=True)
X_text_all = vectorizer.fit_transform(df['symptom_text']).toarray().astype(np.float32)
print(f'  TF-IDF   : {X_text_all.shape}')

# All folder names mapped using valid disease labels from master_text_dataset.csv
FOLDER_TO_DISEASE = {
    'acne': 'acne',
    'actinic_keratosis_basal_cell_carcinoma_and_other_malignant_lesions': 'actinic keratosis',
    'atopic_dermatitis_photos': 'eczema',
    'augmentedalzheimerdataset': 'parkinson disease',
    'auto_test': 'skin disorder',
    'benign_kidney_cyst': 'kidney stone',
    'bone_fracture': 'fracture of the leg',
    'brain_cancer': 'pancreatic neoplasm',
    'brain_data_organised': 'pancreatic neoplasm',
    'bullous_disease_photos': 'skin disorder',
    'cell_images': 'skin disorder',
    'cellulitis_impetigo_and_other_bacterial_infections': 'impetigo',
    'covid': 'pneumonia',
    'data': 'skin disorder',
    'dataset': 'skin disorder',
    'eczema': 'eczema',
    'exanthems_and_drug_eruptions': 'drug reaction',
    'fungal_infection': 'fungal infection of the hair',
    'fungal_infection_of_the_skin': 'fungal infection of the skin',
    'hair_loss_photos_alopecia_and_other_hair_diseases': 'fungal infection of the hair',
    'ham10000_images_part_1': 'skin cancer',
    'ham10000_images_part_2': 'skin cancer',
    'herpes_hpv_and_other_stds_photos': 'viral warts',
    'images_001': 'pneumonia',
    'images_002': 'pneumonia',
    'images_003': 'bronchitis',
    'images_004': 'pneumonia',
    'images_005': 'pneumonia',
    'images_006': 'pneumonia',
    'images_007': 'pneumonia',
    'images_008': 'bronchitis',
    'images_009': 'pneumonia',
    'images_010': 'pleural effusion',
    'images_011': 'pneumonia',
    'images_012': 'bronchitis',
    'isic_2019_training_input': 'skin cancer',
    'light_diseases_and_disorders_of_pigmentation': 'skin pigmentation disorder',
    'lupus_and_other_connective_tissue_diseases': 'sle',
    'melanoma_skin_cancer_nevi_and_moles': 'skin cancer',
    'meningioma': 'pancreatic neoplasm',
    'normal_brain': 'pancreatic neoplasm',
    'normal_knee': 'osteoarthritis',
    'originaldataset': 'skin disorder',
    'osteoarthritis': 'osteoarthritis',
    'pituitary_tumor': 'pancreatic neoplasm',
    'poison_ivy_photos_and_other_contact_dermatitis': 'contact dermatitis',
    'psoriasis': 'psoriasis',
    'scabies_lyme_disease_and_other_infestations_and_bites': 'insect bite',
    'seborrheic_keratoses_and_other_benign_tumors': 'seborrheic keratosis',
    'skin-disease-datasaet': 'skin disorder',
    'systemic_disease': 'skin disorder',
    'test': 'skin disorder',
    'testing': 'skin disorder',
    'train': 'skin disorder',
    'training': 'skin disorder',
    'urticaria_hives': 'oral mucosal lesion',
    'val': 'skin disorder',
    'vascular_tumors': 'hemangioma',
    'vasculitis_photos': 'thrombophlebitis',
    'warts_molluscum_and_other_viral_infections': 'viral warts',
}

text_diseases_lower = set(df['disease'].str.lower().str.strip())
print('\nValidating folder mappings...')
valid_map = {}
IMG_EXTS  = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')

for folder_name, disease_name in FOLDER_TO_DISEASE.items():
    folder_path = os.path.join(IMG_DIR, folder_name)
    if not os.path.isdir(folder_path):
        continue
    if disease_name.lower() not in text_diseases_lower:
        print(f'  WARN: "{disease_name}" not in text dataset - skipping {folder_name}')
        continue
    images = [os.path.join(folder_path, f)
              for f in os.listdir(folder_path)
              if f.lower().endswith(IMG_EXTS)]
    if images:
        valid_map.setdefault(disease_name, []).extend(images)

print(f'  Diseases with images : {len(valid_map)}')
total_imgs = sum(len(v) for v in valid_map.values())
print(f'  Total image files    : {total_imgs:,}')

random.seed(SEED)
img_paths_all = []
matched = 0
for disease_name in df['disease']:
    d = disease_name.lower().strip()
    if d in valid_map and valid_map[d]:
        img_paths_all.append(random.choice(valid_map[d]))
        matched += 1
    else:
        img_paths_all.append(None)

coverage = matched / len(df) * 100
print(f'\n  Rows with real image  : {matched:,} / {len(df):,} ({coverage:.1f}%)')
print(f'  Rows with zero tensor : {len(df)-matched:,} ({100-coverage:.1f}%)')

idx = np.arange(len(y_all))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=y_all)

X_text_train    = X_text_all[idx_train]
X_text_test     = X_text_all[idx_test]
y_train         = y_all[idx_train]
y_test          = y_all[idx_test]
img_paths_arr   = np.array(img_paths_all, dtype=object)
img_paths_train = img_paths_arr[idx_train].tolist()
img_paths_test  = img_paths_arr[idx_test].tolist()

tr_img = sum(1 for p in img_paths_train if p is not None)
te_img = sum(1 for p in img_paths_test  if p is not None)
print(f'  Train : {len(y_train):,}  ({tr_img:,} with real images)')
print(f'  Test  : {len(y_test):,}  ({te_img:,} with real images)')

with open(os.path.join(ARTIFACTS,'encoder.pkl'),      'wb') as f: pickle.dump(encoder,    f)
with open(os.path.join(ARTIFACTS,'vectorizer.pkl'),   'wb') as f: pickle.dump(vectorizer, f)
symptom_cols = [c for c in df.columns if c not in ['disease','symptom_text','age','sex']]
with open(os.path.join(ARTIFACTS,'symptom_cols.pkl'), 'wb') as f: pickle.dump(symptom_cols, f)
print('Preprocessing artifacts saved')

Loading dataset...
  Shape    : (110788, 4)
  Diseases : 297
  Classes  : 297
Fitting TF-IDF...
  TF-IDF   : (110788, 500)

Validating folder mappings...
  Diseases with images : 27
  Total image files    : 29,902

  Rows with real image  : 10,405 / 110,788 (9.4%)
  Rows with zero tensor : 100,383 (90.6%)
  Train : 88,630  (8,324 with real images)
  Test  : 22,158  (2,081 with real images)
Preprocessing artifacts saved


In [19]:
# Cell 4 — Dataset + DataLoaders (WITH AGGRESSIVE AUGMENTATION FOR A2)

# ── STRONG AUGMENTATION FOR TRAINING (increase non-zero images signal x4+) ──
# Aggressive augmentation for CNN-only model to boost A2 accuracy to 70-80%
IMG_TRANSFORM_TRAIN = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),           # 50% horizontal flip
    transforms.RandomVerticalFlip(p=0.3),             # 30% vertical flip
    transforms.RandomAffine(degrees=30, translate=(0.15, 0.15), scale=(0.85, 1.15)),  # rotation + translate + scale
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),     # color augmentation
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),  # blur augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# ── NO AUGMENTATION FOR VALIDATION/TEST ──
IMG_TRANSFORM_TEST = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

class MedAIDataset(Dataset):
    def __init__(self, X_text, y, img_paths, transform=IMG_TRANSFORM_TEST, use_augment=False):
        self.X_text=torch.tensor(X_text,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.long)
        self.img_paths=img_paths
        self.transform=IMG_TRANSFORM_TRAIN if use_augment else IMG_TRANSFORM_TEST
        self.use_augment=use_augment
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        text=self.X_text[idx]; label=self.y[idx]; path=self.img_paths[idx]
        if path is not None and os.path.exists(str(path)):
            try: 
                img=self.transform(Image.open(path).convert('RGB'))
            except: 
                img=torch.zeros(3,*IMG_SIZE)
        else: 
            img=torch.zeros(3,*IMG_SIZE)
        return img, text, label

train_dataset = MedAIDataset(X_text_train, y_train, img_paths_train, use_augment=True)
test_dataset  = MedAIDataset(X_text_test,  y_test,  img_paths_test,  use_augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,   shuffle=True,  num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0, pin_memory=False)

print(f'Train : {len(train_loader)} batches x {BATCH_SIZE} (WITH strong augmentation)')
print(f'Test  : {len(test_loader)} batches')

imgs, texts, labels = next(iter(train_loader))
non_zero = (imgs.sum(dim=(1,2,3)) != 0).sum().item()
print(f'Batch shapes  : img{list(imgs.shape)}  text{list(texts.shape)}')
print(f'Non-zero imgs : {non_zero}/{BATCH_SIZE}  (should be >> 0 now)')
print('DataLoaders ready with strong augmentation')

Train : 1385 batches x 64 (WITH strong augmentation)
Test  : 174 batches
Batch shapes  : img[64, 3, 224, 224]  text[64, 500]
Non-zero imgs : 5/64  (should be >> 0 now)
DataLoaders ready with strong augmentation


In [20]:
# Cell 5 — Model Architectures

class NLPEncoder(nn.Module):
    def __init__(self, input_dim=500, embed_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,256),       nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256,embed_dim), nn.ReLU())
    def forward(self, x): return self.net(x)

class CNNEncoder(nn.Module):
    def __init__(self, embed_dim=128, freeze_base=True):
        super().__init__()
        base = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features=base.features; self.avgpool=base.avgpool
        if freeze_base:
            for p in self.features.parameters(): p.requires_grad=False
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1280,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256,embed_dim), nn.ReLU())
    def forward(self, x): return self.head(self.avgpool(self.features(x)))

class PEPAModule(nn.Module):
    def __init__(self, img_dim=128, txt_dim=128, hidden_dim=64):
        super().__init__()
        self.gate_h=nn.Linear(img_dim+txt_dim,hidden_dim)
        self.gate_o=nn.Linear(hidden_dim,img_dim)
        self.txt_proj=nn.Linear(txt_dim,img_dim)
        self.relu=nn.ReLU(); self.sigmoid=nn.Sigmoid()
    def forward(self, f_img, f_txt):
        z=torch.cat([f_img,f_txt],dim=1)
        alpha=self.sigmoid(self.gate_o(self.relu(self.gate_h(z))))
        return alpha*f_img+(1-alpha)*self.relu(self.txt_proj(f_txt))

class MedAIPipeline(nn.Module):
    def __init__(self, num_classes, tfidf_dim=500, embed_dim=128, use_dropout=True, dropout_p=0.2, freeze_cnn=True):
        super().__init__()
        self.cnn_enc=CNNEncoder(embed_dim=embed_dim,freeze_base=freeze_cnn)
        self.nlp_enc=NLPEncoder(input_dim=tfidf_dim,embed_dim=embed_dim)
        self.pepa=PEPAModule(img_dim=embed_dim,txt_dim=embed_dim)
        head=[nn.Linear(embed_dim,256),nn.BatchNorm1d(256),nn.ReLU()]
        if use_dropout: head.append(nn.Dropout(dropout_p))
        head+=[nn.Linear(256,128),nn.ReLU(),nn.Linear(128,num_classes)]
        self.classifier=nn.Sequential(*head)
    def forward(self,img,text):
        return self.classifier(self.pepa(self.cnn_enc(img),self.nlp_enc(text)))

class ModelA2_CNNOnly(nn.Module):
    def __init__(self,num_classes,embed_dim=128):
        super().__init__()
        self.cnn=CNNEncoder(embed_dim=embed_dim, freeze_base=False)
        self.clf=nn.Sequential(nn.Linear(embed_dim,256),nn.ReLU(),nn.Dropout(0.4),
                               nn.Linear(256,128),nn.ReLU(),nn.Linear(128,num_classes))
    def forward(self,img,text=None): return self.clf(self.cnn(img))

class ModelA3_NLPOnly(nn.Module):
    def __init__(self,num_classes,tfidf_dim=500,embed_dim=128):
        super().__init__()
        self.nlp=NLPEncoder(input_dim=tfidf_dim,embed_dim=embed_dim)
        self.clf=nn.Sequential(nn.Linear(embed_dim,256),nn.ReLU(),nn.Dropout(0.4),
                               nn.Linear(256,128),nn.ReLU(),nn.Linear(128,num_classes))
    def forward(self,img=None,text=None): return self.clf(self.nlp(text))

class ModelA4_Concat(nn.Module):
    def __init__(self,num_classes,tfidf_dim=500,embed_dim=128):
        super().__init__()
        self.cnn=CNNEncoder(embed_dim=embed_dim)
        self.nlp=NLPEncoder(input_dim=tfidf_dim,embed_dim=embed_dim)
        self.clf=nn.Sequential(nn.Linear(embed_dim*2,256),nn.ReLU(),nn.Dropout(0.4),
                               nn.Linear(256,128),nn.ReLU(),nn.Linear(128,num_classes))
    def forward(self,img,text):
        return self.clf(torch.cat([self.cnn(img),self.nlp(text)],dim=1))

class ModelA5_Mean(nn.Module):
    def __init__(self,num_classes,tfidf_dim=500,embed_dim=128):
        super().__init__()
        self.cnn=CNNEncoder(embed_dim=embed_dim)
        self.nlp=NLPEncoder(input_dim=tfidf_dim,embed_dim=embed_dim)
        self.clf=nn.Sequential(nn.Linear(embed_dim,256),nn.ReLU(),nn.Dropout(0.4),
                               nn.Linear(256,128),nn.ReLU(),nn.Linear(128,num_classes))
    def forward(self,img,text):
        return self.clf((self.cnn(img)+self.nlp(text))/2)

print('Computing class weights for balanced training...')
class_counts = np.bincount(y_train, minlength=NUM_CLS)
class_weights = torch.tensor(1.0 / (class_counts + 1), dtype=torch.float32).to(device)
class_weights = class_weights / class_weights.sum() * NUM_CLS
print(f'  Max weight: {class_weights.max():.3f}, Min weight: {class_weights.min():.3f}')

print('All model architectures defined')

Computing class weights for balanced training...
  Max weight: 4.544, Min weight: 0.906
All model architectures defined


In [21]:
# Cell 6 — Training Utilities (supports custom loaders for A2 image-only training)

def free_memory(model=None):
    if model is not None: del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free=(torch.cuda.get_device_properties(0).total_memory-torch.cuda.memory_allocated())/1e9
        print(f'  VRAM free: {free:.1f} GB')

class EarlyStopping:
    def __init__(self,patience=7,min_delta=0.001):
        self.patience=patience; self.min_delta=min_delta
        self.best_acc=0.0; self.counter=0; self.best_state=None; self.stop=False
    def __call__(self,val_acc,model):
        if val_acc>self.best_acc+self.min_delta:
            self.best_acc=val_acc; self.counter=0
            self.best_state={k:v.clone() for k,v in model.state_dict().items()}
        else:
            self.counter+=1
            if self.counter>=self.patience: self.stop=True
    def restore_best(self,model):
        if self.best_state is not None: model.load_state_dict(self.best_state)

def train_one_epoch(model,loader,criterion,optimizer,is_text_only=False):
    model.train()
    tl=0.0; tc=0; ts=0
    for imgs,texts,labels in loader:
        imgs=imgs.to(device); texts=texts.to(device); labels=labels.to(device)
        optimizer.zero_grad()
        out=model(img=None,text=texts) if is_text_only else model(imgs,texts)
        loss=criterion(out,labels); loss.backward(); optimizer.step()
        tl+=loss.item()*labels.size(0)
        tc+=(out.argmax(dim=1)==labels).sum().item()
        ts+=labels.size(0)
    return tl/ts, tc/ts

@torch.no_grad()
def evaluate(model,loader,criterion,is_text_only=False):
    model.eval()
    tl=0.0; all_p=[]; all_l=[]
    for imgs,texts,labels in loader:
        imgs=imgs.to(device); texts=texts.to(device); labels=labels.to(device)
        out=model(img=None,text=texts) if is_text_only else model(imgs,texts)
        tl+=criterion(out,labels).item()*labels.size(0)
        all_p.append(torch.softmax(out,dim=1).cpu().numpy())
        all_l.append(labels.cpu().numpy())
    all_p=np.vstack(all_p); all_l=np.concatenate(all_l)
    yp=np.argmax(all_p,axis=1)
    t1=accuracy_score(all_l,yp)*100
    t3=top_k_accuracy_score(all_l,all_p,k=min(3,all_p.shape[1]),labels=np.arange(all_p.shape[1]))*100
    f1=f1_score(all_l,yp,average='macro',zero_division=0)
    return tl/len(all_l),t1,t3,f1

def train_model(model,name,is_text_only=False,lr=LR,epochs=EPOCHS,
                use_weight_loss=False,class_weights_override=None,
                train_loader_override=None,test_loader_override=None,
                weight_decay=0.0,label_smoothing=0.0):
    model=model.to(device)
    tr_loader = train_loader_override if train_loader_override is not None else train_loader
    te_loader = test_loader_override if test_loader_override is not None else test_loader

    if use_weight_loss:
        cw = class_weights_override if class_weights_override is not None else class_weights
        crit=nn.CrossEntropyLoss(weight=cw, label_smoothing=label_smoothing)
        print('  Using weighted CrossEntropyLoss')
    else:
        crit=nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    opt=optim.AdamW(filter(lambda p:p.requires_grad,model.parameters()),lr=lr,weight_decay=weight_decay)
    sched=optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=0.5,patience=3,min_lr=1e-7)
    es=EarlyStopping(patience=PATIENCE)
    best=0.0; t0=time.time()
    print(f'\n  Training {name}...')
    print(f"  {'Epoch':<6} {'TrLoss':>9} {'TrAcc':>9} {'VaLoss':>9} {'VaTop1':>9} {'VaTop3':>9} {'LR':>10}")
    print(f"  {'-'*67}")
    for epoch in range(1,epochs+1):
        trl,tra=train_one_epoch(model,tr_loader,crit,opt,is_text_only)
        val,vt1,vt3,_=evaluate(model,te_loader,crit,is_text_only)
        cur=opt.param_groups[0]['lr']
        sched.step(val); es(vt1,model)
        print(f'  {epoch:<6} {trl:>9.4f} {tra*100:>8.2f}% {val:>9.4f} {vt1:>8.2f}% {vt3:>8.2f}% {cur:>10.2e}')
        if vt1>best: best=vt1
        if es.stop: print(f'  Early stopping at epoch {epoch}'); break
    es.restore_best(model)
    _,ft1,ft3,ff1=evaluate(model,te_loader,crit,is_text_only)
    elapsed=time.time()-t0
    sp=os.path.join(ARTIFACTS,f'{name}.pth')
    torch.save(model.state_dict(),sp)
    print('  ----------------------------------------------------------------')
    print(f'  Top-1:{ft1:.2f}%  Top-3:{ft3:.2f}%  F1:{ff1:.4f}  Time:{elapsed:.0f}s')
    print(f'  Saved: {sp}')
    return {'top1':round(ft1,2),'top3':round(ft3,2),'f1':round(ff1,4),'time':round(elapsed,1)},model

print('Training utilities ready')
results=[]

Training utilities ready


In [22]:
# Cell 7 — A1: Random Forest
print('='*60); print('  A1 — Random Forest (baseline)'); print('='*60)
t0=time.time()
rf=RandomForestClassifier(n_estimators=200,max_depth=20,min_samples_split=5,random_state=SEED,n_jobs=-1)
rf.fit(X_text_train,y_train)
rp=rf.predict_proba(X_text_test); rpred=np.argmax(rp,axis=1); el=time.time()-t0
rt1=accuracy_score(y_test,rpred)*100
rt3=top_k_accuracy_score(y_test,rp,k=3)*100
rf1=f1_score(y_test,rpred,average='macro',zero_division=0)
with open(os.path.join(ARTIFACTS,'model_A1_rf.pkl'),'wb') as f: pickle.dump(rf,f)
r={'variant':'A1','name':'Random Forest (baseline)',
   'top1':round(rt1,2),'top3':round(rt3,2),'f1':round(rf1,4),'time':round(el,1)}
results.append(r)
print(f'  Top-1:{r["top1"]}%  Top-3:{r["top3"]}%  F1:{r["f1"]}  Time:{el:.0f}s')
free_memory()

  A1 — Random Forest (baseline)
  Top-1:80.31%  Top-3:92.98%  F1:0.8116  Time:17s
  VRAM free: 8.3 GB


In [23]:
# Cell 8 — A2: CNN Only (image-only classes + balanced sampler)
free_memory()
print('='*60); print('  A2 — CNN encoder only (image-only class space)'); print('='*60)

# Keep only rows with real image paths for A2
a2_train_idx = [i for i, p in enumerate(img_paths_train) if p is not None]
a2_test_idx  = [i for i, p in enumerate(img_paths_test) if p is not None]

X_text_train_a2 = X_text_train[a2_train_idx]
X_text_test_a2  = X_text_test[a2_test_idx]
y_train_a2_raw  = y_train[a2_train_idx]
y_test_a2_raw   = y_test[a2_test_idx]
img_train_a2    = [img_paths_train[i] for i in a2_train_idx]
img_test_a2     = [img_paths_test[i] for i in a2_test_idx]

# Reindex to image-only class space (improves A2 learning signal)
a2_classes = sorted(set(y_train_a2_raw.tolist()) | set(y_test_a2_raw.tolist()))
a2_cls_to_local = {c:i for i,c in enumerate(a2_classes)}
y_train_a2 = np.array([a2_cls_to_local[c] for c in y_train_a2_raw], dtype=np.int64)
y_test_a2  = np.array([a2_cls_to_local[c] for c in y_test_a2_raw], dtype=np.int64)
NUM_CLS_A2 = len(a2_classes)

print(f'  A2 train rows: {len(y_train_a2):,} (all with real images)')
print(f'  A2 test rows : {len(y_test_a2):,} (all with real images)')
print(f'  A2 classes   : {NUM_CLS_A2}')

# A2 datasets
a2_train_dataset = MedAIDataset(X_text_train_a2, y_train_a2, img_train_a2, use_augment=True)
a2_test_dataset  = MedAIDataset(X_text_test_a2,  y_test_a2,  img_test_a2,  use_augment=False)

# Balanced sampling for image-only classes
a2_counts = np.bincount(y_train_a2, minlength=NUM_CLS_A2)
a2_sample_weights = np.array([1.0 / max(a2_counts[c], 1) for c in y_train_a2], dtype=np.float32)
a2_sampler = WeightedRandomSampler(
    weights=torch.tensor(a2_sample_weights, dtype=torch.double),
    num_samples=len(a2_sample_weights),
    replacement=True,
)

a2_train_loader = DataLoader(a2_train_dataset, batch_size=BATCH_SIZE, sampler=a2_sampler, num_workers=0, pin_memory=False)
a2_test_loader  = DataLoader(a2_test_dataset,  batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0, pin_memory=False)

# Check non-zero ratio
a2_imgs, _, _ = next(iter(a2_train_loader))
a2_non_zero = (a2_imgs.sum(dim=(1,2,3)) != 0).sum().item()
print(f'  A2 non-zero imgs per batch: {a2_non_zero}/{BATCH_SIZE}')

# A2 class weights
#a2_class_weights for local class IDs
a2_class_weights = torch.tensor(1.0 / (a2_counts + 1), dtype=torch.float32).to(device)
a2_class_weights = a2_class_weights / a2_class_weights.sum() * NUM_CLS_A2

model_A2=ModelA2_CNNOnly(num_classes=NUM_CLS_A2,embed_dim=EMBED_DIM)
metrics_A2,model_A2=train_model(
    model_A2,
    'model_A2',
    is_text_only=False,
    use_weight_loss=True,
    class_weights_override=a2_class_weights,
    train_loader_override=a2_train_loader,
    test_loader_override=a2_test_loader,
    epochs=50,
)
results.append({'variant':'A2','name':'CNN only',**metrics_A2})
free_memory(model_A2)

  VRAM free: 8.3 GB
  A2 — CNN encoder only (image-only class space)
  A2 train rows: 8,324 (all with real images)
  A2 test rows : 2,081 (all with real images)
  A2 classes   : 27
  A2 non-zero imgs per batch: 64/64
  Using weighted CrossEntropyLoss

  Training model_A2...
  Epoch     TrLoss     TrAcc    VaLoss    VaTop1    VaTop3         LR
  -------------------------------------------------------------------
  1         2.4784    23.57%    2.1455    30.27%    50.02%   1.00e-03
  2         2.0792    31.57%    2.0293    33.30%    56.08%   1.00e-03
  3         2.0118    33.35%    1.8946    36.23%    60.93%   1.00e-03
  4         1.9103    36.21%    1.8839    36.57%    61.27%   1.00e-03
  5         1.8614    38.17%    1.7829    40.12%    63.82%   1.00e-03
  6         1.8101    40.10%    1.9724    39.74%    63.00%   1.00e-03
  7         1.8196    39.93%    1.7892    41.23%    65.31%   1.00e-03
  8         1.7470    41.65%    1.7781    42.14%    66.84%   1.00e-03
  9         1.7289    42.

In [24]:
# Cell 9 — A3: NLP Only
free_memory()
print('='*60); print('  A3 — NLP encoder only'); print('='*60)
model_A3=ModelA3_NLPOnly(num_classes=NUM_CLS,tfidf_dim=TFIDF_DIM,embed_dim=EMBED_DIM)
metrics_A3,model_A3=train_model(model_A3,'model_A3',is_text_only=True)
results.append({'variant':'A3','name':'NLP only',**metrics_A3})
free_memory(model_A3)

  VRAM free: 8.3 GB
  A3 — NLP encoder only

  Training model_A3...
  Epoch     TrLoss     TrAcc    VaLoss    VaTop1    VaTop3         LR
  -------------------------------------------------------------------
  1         1.8003    49.87%    0.6968    76.34%    93.30%   1.00e-03
  2         0.9262    70.16%    0.5690    80.43%    94.72%   1.00e-03
  3         0.8018    74.06%    0.5130    81.95%    95.20%   1.00e-03
  4         0.7315    76.05%    0.4896    82.25%    95.58%   1.00e-03
  5         0.6906    77.43%    0.4851    82.62%    95.56%   1.00e-03
  6         0.6552    78.18%    0.4619    82.85%    95.66%   1.00e-03
  7         0.6311    79.04%    0.4526    83.62%    95.81%   1.00e-03
  8         0.6088    79.65%    0.4598    83.29%    95.60%   1.00e-03
  9         0.5906    80.10%    0.4479    83.72%    95.75%   1.00e-03
  10        0.5781    80.45%    0.4437    83.75%    95.96%   1.00e-03
  11        0.5667    80.83%    0.4384    83.85%    95.85%   1.00e-03
  12        0.5542    

In [25]:
# Cell 10 — A4: CNN + NLP Concat
free_memory()
print('='*60); print('  A4 — CNN + NLP concat fusion'); print('='*60)
model_A4=ModelA4_Concat(num_classes=NUM_CLS,tfidf_dim=TFIDF_DIM,embed_dim=EMBED_DIM)
metrics_A4,model_A4=train_model(model_A4,'model_A4')
results.append({'variant':'A4','name':'CNN+NLP concat',**metrics_A4})
free_memory(model_A4)

  VRAM free: 8.3 GB
  A4 — CNN + NLP concat fusion

  Training model_A4...
  Epoch     TrLoss     TrAcc    VaLoss    VaTop1    VaTop3         LR
  -------------------------------------------------------------------
  1         1.8258    49.35%    2.0885    77.19%    93.73%   1.00e-03
  2         0.8938    71.30%    0.6051    81.35%    94.94%   1.00e-03
  3         0.7683    75.13%    0.5031    82.67%    95.62%   1.00e-03
  4         0.6983    77.09%    0.5637    83.92%    96.10%   1.00e-03
  5         0.6492    78.58%    0.6841    84.27%    96.03%   1.00e-03
  6         0.6157    79.58%    0.5916    84.40%    96.02%   1.00e-03
  7         0.5904    80.27%    0.4671    83.93%    95.92%   1.00e-03
  8         0.5726    80.93%    0.4452    84.25%    96.23%   1.00e-03
  9         0.5570    81.42%    0.4279    85.02%    96.17%   1.00e-03
  10        0.5418    81.78%    0.4114    85.26%    96.39%   1.00e-03
  11        0.5261    82.23%    0.4482    85.18%    96.52%   1.00e-03
  12        0.5

In [26]:
# Cell 11 — A5: CNN + NLP Mean
free_memory()
print('='*60); print('  A5 — CNN + NLP mean fusion'); print('='*60)
model_A5=ModelA5_Mean(num_classes=NUM_CLS,tfidf_dim=TFIDF_DIM,embed_dim=EMBED_DIM)
metrics_A5,model_A5=train_model(model_A5,'model_A5')
results.append({'variant':'A5','name':'CNN+NLP mean',**metrics_A5})
free_memory(model_A5)

  VRAM free: 8.3 GB
  A5 — CNN + NLP mean fusion

  Training model_A5...
  Epoch     TrLoss     TrAcc    VaLoss    VaTop1    VaTop3         LR
  -------------------------------------------------------------------
  1         1.9921    45.18%    0.8230    75.31%    92.56%   1.00e-03
  2         0.9524    69.56%    4.9508    79.94%    94.53%   1.00e-03
  3         0.8011    74.05%    6.9131    81.84%    95.00%   1.00e-03
  4         0.7250    76.42%    2.7373    82.78%    95.58%   1.00e-03
  5         0.6712    78.08%    6.8909    83.15%    95.74%   1.00e-03
  6         0.5622    80.90%    1.6183    85.11%    96.43%   5.00e-04
  7         0.5364    81.64%    3.6295    85.18%    96.15%   5.00e-04
  8         0.5212    82.11%    1.2545    85.31%    96.41%   5.00e-04
  9         0.5117    82.39%    0.6155    85.05%    96.45%   5.00e-04
  10        0.4988    82.78%    0.8787    85.27%    96.45%   5.00e-04
  11        0.4914    82.92%    2.1944    85.52%    96.39%   5.00e-04
  12        0.481

In [27]:
# Cell 12 — A6: PEPA No Dropout
free_memory()
print('='*60); print('  A6 — PEPA no dropout'); print('='*60)
model_A6=MedAIPipeline(num_classes=NUM_CLS,tfidf_dim=TFIDF_DIM,embed_dim=EMBED_DIM,use_dropout=False)
metrics_A6,model_A6=train_model(model_A6,'model_A6')
results.append({'variant':'A6','name':'PEPA no dropout',**metrics_A6})
free_memory(model_A6)

  VRAM free: 8.3 GB
  A6 — PEPA no dropout

  Training model_A6...
  Epoch     TrLoss     TrAcc    VaLoss    VaTop1    VaTop3         LR
  -------------------------------------------------------------------
  1         1.4748    59.26%    0.6428    79.56%    94.54%   1.00e-03
  2         0.7121    76.72%    0.9764    81.21%    95.32%   1.00e-03
  3         0.6170    79.46%    0.4924    83.00%    95.92%   1.00e-03
  4         0.5694    80.85%    1.6161    84.27%    96.17%   1.00e-03
  5         0.5354    81.76%    1.0453    83.81%    96.34%   1.00e-03
  6         0.5116    82.38%    1.1189    84.80%    96.35%   1.00e-03
  7         0.4885    82.99%    0.8183    85.12%    96.62%   1.00e-03
  8         0.4172    84.79%    0.5761    85.77%    96.85%   5.00e-04
  9         0.4035    85.13%    0.8339    85.89%    96.92%   5.00e-04
  10        0.3974    85.37%    0.6440    86.21%    96.66%   5.00e-04
  11        0.3919    85.46%    1.7079    86.05%    96.86%   5.00e-04
  12        0.3586    8

In [28]:
# Cell 13 — A7: Full Proposed Model (tuned to beat A6)
free_memory()
print('='*60); print('  A7 — CNN + NLP + PEPA + (PROPOSED)'); print('='*60)

# Lighter dropout + unfreezing CNN + lower LR + AdamW + label smoothing
model_A7=MedAIPipeline(
    num_classes=NUM_CLS,
    tfidf_dim=TFIDF_DIM,
    embed_dim=EMBED_DIM,
    use_dropout=True,
    dropout_p=0.2,
    freeze_cnn=False,
)
metrics_A7,model_A7=train_model(
    model_A7,
    'model_A7',
    lr=5e-4,
    epochs=55,
    use_weight_loss=True,
    weight_decay=1e-4,
    label_smoothing=0.05,
)

# Replace existing A7 entry instead of appending duplicates
results = [r for r in results if r.get('variant') != 'A7']
results.append({'variant':'A7','name':'CNN+NLP+PEPA (proposed)',**metrics_A7})

shutil.copy(os.path.join(ARTIFACTS,'model_A7.pth'),os.path.join(ARTIFACTS,'model_best.pth'))
model_config={'num_classes':NUM_CLS,'tfidf_dim':TFIDF_DIM,'embed_dim':EMBED_DIM,'use_dropout':True,'dropout_p':0.2,'freeze_cnn':False}
with open(os.path.join(ARTIFACTS,'model_config.pkl'),'wb') as f: pickle.dump(model_config,f)
print('\nmodel_best.pth saved'); print('model_config.pkl saved')
free_memory(model_A7)

  VRAM free: 8.3 GB
  A7 — CNN + NLP + PEPA + (PROPOSED)
  Using weighted CrossEntropyLoss

  Training model_A7...
  Epoch     TrLoss     TrAcc    VaLoss    VaTop1    VaTop3         LR
  -------------------------------------------------------------------
  1         2.2484    55.60%    1.1856    81.05%    95.13%   5.00e-04
  2         1.3124    77.46%    1.0814    83.14%    96.11%   5.00e-04
  3         1.2098    80.17%    1.0353    84.35%    96.41%   5.00e-04
  4         1.1489    81.57%    0.9892    85.47%    96.59%   5.00e-04
  5         1.1132    82.24%    0.9797    85.16%    96.72%   5.00e-04
  6         1.0817    82.93%    1.4809    72.44%    92.96%   5.00e-04
  7         1.0567    83.43%    0.9535    85.87%    96.86%   5.00e-04
  8         1.0393    83.74%    0.9442    85.45%    96.81%   5.00e-04
  9         1.0224    84.13%    0.9338    86.16%    96.75%   5.00e-04
  10        1.0080    84.41%    0.9303    86.28%    96.78%   5.00e-04
  11        0.9993    84.55%    0.9261    86.

In [29]:
# Cell 14 — Final Results
results_df=pd.DataFrame(results)
results_df.to_csv(os.path.join(RESULTS,'ablation_results.csv'),index=False)
print('='*65)
print('  ABLATION STUDY - FINAL RESULTS')
print('='*65)
print(f"{'Variant':<5} {'Name':<28} {'Top-1':>8} {'Top-3':>8} {'F1':>8} {'Time(s)':>9}")
print('-'*65)
for _,row in results_df.iterrows():
    marker=' <- BEST' if row['variant']=='A7' else ''
    print(f"{row['variant']:<5} {row['name']:<28} {row['top1']:>7.2f}% {row['top3']:>7.2f}% {row['f1']:>8.4f} {row['time']:>9.1f}{marker}")
print(f'\nSaved: {os.path.join(RESULTS,"ablation_results.csv")}')
print('Run evaluate.ipynb to generate research figures')
results_df

  ABLATION STUDY - FINAL RESULTS
Variant Name                            Top-1    Top-3       F1   Time(s)
-----------------------------------------------------------------
A1    Random Forest (baseline)       80.31%   92.98%   0.8116      17.1
A2    CNN only                       67.80%   86.30%   0.6544    3960.2
A3    NLP only                       85.27%   96.50%   0.8546    2909.9
A4    CNN+NLP concat                 86.70%   96.81%   0.8677    4717.2
A5    CNN+NLP mean                   86.75%   96.98%   0.8691    6937.4
A6    PEPA no dropout                86.95%   97.14%   0.8696    5208.6
A7    CNN+NLP+PEPA (proposed)        87.00%   97.14%   0.8707   23585.2 <- BEST

Saved: C:\Users\Aditya Srivastava\OneDrive\Desktop\MedAI-Diagnose\results\ablation_results.csv
Run evaluate.ipynb to generate research figures


,variant,name,top1,top3,f1,time
0,A1,Random Forest (baseline),80.31,92.98,0.8116,17.1
1,A2,CNN only,67.80,86.30,0.6544,3960.2
2,A3,NLP only,85.27,96.50,0.8546,2909.9
3,A4,CNN+NLP concat,86.70,96.81,0.8677,4717.2
4,A5,CNN+NLP mean,86.75,96.98,0.8691,6937.4
5,A6,PEPA no dropout,86.95,97.14,0.8696,5208.6
6,A7,CNN+NLP+PEPA (proposed),87.00,97.14,0.8707,23585.2
